### Tools

###### Models cam request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:

###### 1. A schema including the name of the tool, a description, and/or arguments definitions ( often a JSON schema)
###### 2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROOG_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='**Short answer:**  \nParrots “talk” because they are exceptionally good vocal mimics, and in a captive environment they learn to copy the sounds that humans make, especially when those sounds are reinforced with attention, food, or social interaction. Their ability to do this is rooted in their anatomy, brain organization, and social nature—not because they understand human language the way we do.\n\n---\n\n## 1. The anatomy that makes mimicry possible  \n\n| Feature | What it is | Why it matters for mimicry |\n|---------|------------|---------------------------|\n| **Syrinx** (bird vocal organ) | Located at the base of the trachea, the syrinx contains two sets of vibrating membranes (labia) that can be controlled independently. | Allows a single bird to produce two different tones at once and to make rapid, precise changes in pitch and timbre—far more flexible than the human larynx. |\n| **Highly developed respiratory control** | Birds have a system of air sacs that

In [3]:
### creating tools

from langchain.tools import tool

### We will use @Tool decorator and that function become tool

@tool
def get_weather(location:str)-> str:
    """
     Get the weather at a location

    """
    return  f"It's Sunny in {location}"


### Bind the tools
model_with_tools = model.bind_tools([get_weather])

In [6]:
### Invoke the tools

response = model_with_tools.invoke("What is the weather in Kansas?")
print(response)
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks "What is the weather in Kansas?" We need to fetch weather. Likely need to call get_weather with location "Kansas". Use function call.', 'tool_calls': [{'id': 'fc_357bb1d6-a033-4824-bb7d-df3f7f00ff86', 'function': {'arguments': '{"location":"Kansas"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 127, 'total_tokens': 188, 'completion_time': 0.128092377, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.006050998, 'prompt_tokens_details': None, 'queue_time': 0.344164349, 'total_time': 0.134143375}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_be28f5c9f2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a10749-86ef-7c11-8c41-3743c89559f2-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Kansas'}, 'id': 'fc_357bb1d6-a033-4824-bb7

### Tool Execution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content":"What's the weather in Kansas?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The Current weather in Kansas is 72 degree F and Sunny"